In [5]:
import json
import os
import sys
from pathlib import Path
from typing import Literal

import pandas as pd
from openai import AsyncOpenAI, APITimeoutError
from pydantic import BaseModel, Field
from pydantic.type_adapter import TypeAdapter
from sklearn.metrics import classification_report
from tqdm.asyncio import tqdm

sys.path.append("../..")

from climateguard.models import Article

In [2]:
client = AsyncOpenAI(timeout=120) # Default is 10 minutes, OpenAI frequently times out for no reason


class Transcript(BaseModel):
    url: str | None = None
    title: str | None = None
    content: str
    language: Literal["English", "French", "Latvian"]
    is_related_to_climate: bool
    translation: str | None = None

    @staticmethod
    def from_json(filepath: str) -> list["Transcript"]:
        with open(filepath) as f:
            return TypeAdapter(list[Transcript]).validate_json(f.read())

## Create a benchmark dataset

In [3]:
def save_transcripts(
    transcripts: Transcript | list[Transcript],
    folder: Literal["ok", "not_ok"],
    filename: str,
) -> None:
    transcripts = transcripts if isinstance(transcripts, list) else [transcripts]
    with open(Path("data") / folder / f"{filename}.json", "w") as f:
        json.dump(
            [transcript.model_dump(exclude_none=True) for transcript in transcripts],
            f,
            indent=2,
        )

### Manually curated data

In [4]:
def parse_dailymotion_transcript(filepath: str) -> Transcript:
    with open(filepath) as f:
        data = json.load(f)

    texts = [
        node["node"]["text"].strip()
        for node in data["data"]["video"]["transcript"]["edges"]
    ]
    text = " ".join(" ".join(texts).split()).strip()

    transcript = Transcript(
        url=f'https://www.dailymotion.com/video/{data["data"]["video"]["xid"]}',
        title=data["data"]["video"]["title"],
        content=text,
        is_related_to_climate=True,
        language="French",
    )

    return transcript


save_transcripts(
    parse_dailymotion_transcript(
        "data/raw/europe_1_chateau_versailles_transcript_ok.json"
    ),
    "ok",
    "europe_1_chateau_versailles",
)
save_transcripts(
    parse_dailymotion_transcript(
        "data/raw/france_bleu_transcript_legislatives_not_ok.json"
    ),
    "not_ok",
    "france_bleu_legislatives",
)

In [5]:
raptor = pd.read_csv("data/raw/youtube_transcript_raptor_not_ok.csv")
transcript = Transcript(
    url="https://www.youtube.com/watch?v=6OenzW3ODsI",
    title="RÉCHAUFFEMENT CLIMATIQUE : DÉCRYPTAGE D'UNE ARNAQUE MONDIALE",
    content=" ".join(" ".join(raptor["text"].str.strip().tolist()).split()),
    language="French",
    is_related_to_climate=True,
)
save_transcripts(transcript, "not_ok", "youtube_transcript_raptor")

### Scraped news articles

In [6]:
articles = Article.from_json("data/raw/scraped_articles.json")
transcripts = [
    Transcript(
        url=article.url,
        title=article.title,
        content=article.content,
        language="Latvian",
        is_related_to_climate=True,
    )
    for article in articles
]
save_transcripts(transcripts, "ok", "scraped_news_articles")

### Latvia TV

In [7]:
df = pd.read_parquet("data/raw/latvia_tv_with_scores.parquet")
transcripts = [
    Transcript(
        url=webpage_url,
        content=raw_text,
        language="Latvian",
        translation=translation,
        is_related_to_climate=is_environmental,
    )
    for webpage_url, raw_text, translation, is_environmental in df[
        ["webpage_url", "raw_text", "translation", "is_environmental"]
    ].itertuples(index=False)
]
save_transcripts(transcripts, "ok", "latvia_tv")

### Generate synthetic news articles

In [17]:
class SyntheticArticle(BaseModel):
    title: str
    content: str


async def generate_synthetic_article(claim: str) -> SyntheticArticle:
    response = await client.beta.chat.completions.parse(
        messages=[
            {
                "role": "system",
                "content": """You are an AI tasked with crafting articles that reflect climate change skepticism. Your objective is to generate content rooted in user-specific assertions about climate change. Each article should present a perspective that questions the immediacy and severity of climate change, challenging prevailing scientific views while incorporating elements characteristic of broader skepticism towards established science—such as flat-earth and anti-vaccine viewpoints.

Highlight a mindset of questioning mainstream science, expressing skepticism towards established experts, and exploring alternative theories. Employ convincing language, personal stories, and references to historical trends or unconventional theories to bolster the skeptical viewpoint. Aim to persuade readers that concerns about climate change are either overstated or manageable without drastic measures.""",
            },
            {
                "role": "user",
                "content": f"Claim: {claim}",
            },
        ],
        response_format=SyntheticArticle,
        temperature=1,
        model="gpt-4o-mini",
    )
    return response.choices[0].message.parsed


tasks = []
claims = [
    "Climate change might be happening, but it's not urgent; we still have plenty of time to adapt.",
    "Electric vehicles still rely on fossil fuels for electricity, so they're not really helping the environment.",
    "Planting more trees will easily offset all of our carbon emissions.",
    "We can solve climate change just by using more technology without needing to change our lifestyles.",
    "Plastic recycling is efficient enough to manage pollution, so there's no need to reduce plastic use.",
    "Increased carbon dioxide is good for plants and agriculture, so climate change could actually benefit us.",
    "The Earth naturally balances itself, so human impact isn't significant in the long run.",
    "It's more important to focus on economic growth than to worry about environmental regulations.",
    "Natural disasters are part of the Earth's natural cycles, not related to climate change.",
    "Climate models are too complex and unreliable to make accurate predictions.",
    "There's no consensus among scientists about the severity of climate change.",
    "Historically, the Earth has always gone through warm and cool periods.",
    "Polar bear populations are actually increasing, not decreasing.",
    "The effects of climate change are over-exaggerated by the media for sensationalism.",
    "Global warming stopped in the late 1990s; recent temperature rise is negligible.",
    "Carbon taxes will hurt the economy without significantly impacting climate change.",
    "Climate science is too politicized to be considered objective.",
    "The cost of mitigating climate change outweighs the potential benefits.",
    "Most climate research is funded by groups with agendas, skewing results.",
    "The sun's activity is responsible for climate changes, not human activity.",
    "There is not enough long-term data to prove climate change is happening.",
    "Sea level rise predictions are exaggerated and won't impact us significantly.",
    "Climate policies could lead to energy shortages and increased poverty.",
    "Technological innovation will automatically solve climate problems.",
    "There are more pressing issues than climate change right now.",
    "Climate change initiatives are a means for governments to increase control.",
    "Oceans have vast capacity to absorb CO2, so human emissions are insignificant.",
    "Volcanic eruptions emit more CO2 than human activities.",
    "Climate change hasn't affected our daily lives significantly, so it's not urgent.",
    "Prioritizing climate change takes resources away from health and education.",
    "Extinction rates aren't accelerating, they are consistent with natural cycles.",
    "The benefits of fossil fuels outweigh their environmental costs.",
    "Sea ice levels have stabilized; there's no need for alarm.",
    "Fossil fuels are necessary for economic stability and growth.",
    "The Paris Agreement is ineffective and too costly for economies.",
    "There's no empirical data to support the claim that storms are worsening.",
    "Human adaptability will naturally overcome climate-related challenges.",
    "Environmental regulations stifle innovation and entrepreneurship.",
    "Food production won't be significantly impacted by climate change.",
    "Corporations are unfairly blamed for environmental degradation.",
    "The benefits of a warmer climate, such as longer growing seasons, are ignored.",
    "Energy consumption reductions hinder technological progress.",
    "Global migration patterns won't be affected dramatically by climate change.",
    "Coral bleaching events are reversible and not solely linked to climate change.",
    "Plants will adapt to increased CO2 levels without human interference.",
    "Geoengineering solutions will mitigate climate risks without lifestyle change.",
    "The narrative on climate change is driven by fear rather than facts.",
    "Methane emissions from livestock are overstated as a climate driver.",
    "Antarctic ice levels show little change, contradicting warming claims.",
    "Human-related activity contributes minimally to historical climate trends.",
    "The resilience of natural ecosystems is underestimated in climate models.",
    "Technological advancements in agriculture will offset negative climate effects.",
    "The impact of deforestation is reversible and overstated.",
    "Climate change is a luxury issue for wealthy nations, not a global crisis.",
    "The link between fossil fuel use and extreme weather lacks concrete evidence.",
    "Reducing fossil fuel use will lead to job losses and economic downturns.",
    "Public perception of climate urgency is manipulated by advocacy groups.",
    "Increased hurricane activity is part of natural oceanic cycles.",
    "Most CO2 in the atmosphere is naturally occurring, not human-made.",
    "Current sea level rise projections are inconsistent and often contradictory.",
    "Consumer behavior doesn't impact climate outcomes.",
    "The uncertainty in climate sensitivity reduces the urgency for action.",
    "Flooding incidents are linked to poor infrastructure, not just climate change.",
    "National energy independence requires maintaining fossil fuel industries.",
]
synthetic_articles: list[SyntheticArticle] = await tqdm.gather(
    *[generate_synthetic_article(claim) for claim in claims],
    desc="Generating synthetic articles",
)
transcripts = [
    Transcript(
        title=synthetic_article.title,
        content=synthetic_article.content,
        language="English",
        is_related_to_climate=True,
    )
    for synthetic_article in synthetic_articles
]
save_transcripts(transcripts, "not_ok", "synthetic_news_articles")

Generating synthetic articles: 100%|██████████| 64/64 [00:13<00:00,  4.88it/s]


## Classify transcripts

### Load benchmark dataset

In [3]:
transcripts = []
for folder in ["ok", "not_ok"]:
    for filename in os.listdir(f"data/{folder}"):
        for transcript in Transcript.from_json(f"data/{folder}/{filename}"):
            transcripts.append(
                {
                    "source": filename.replace(".json", ""),
                    **transcript.model_dump(),
                    "y_true": folder == "not_ok",
                }
            )
df = pd.DataFrame(transcripts)

# # Sample fewer rows
# df = df[df["is_related_to_climate"]]
# # Sample at most 100 rows for each category in the 'source' column for quick tests
# df = (
#     df.groupby("source")
#     .apply(lambda x: x.sample(n=min(len(x), 500), random_state=1), include_groups=False)
#     .reset_index(level=0)
#     .reset_index(drop=True)
# )
df

,source,url,title,content,language,is_related_to_climate,translation,y_true
0,europe_1_chateau_versailles,https://www.dailymotion.com/video/x8jxqc9,Climat : le château de Versailles victime du r...,mais on découvre que c'est aussi un danger pou...,French,True,None,False
1,latvia_tv,https://www.youtube.com/watch?v=-0-x5hyqIc0,None,"Saskaņā ar pēdējiem SKDS reitingiem, kas publi...",Latvian,False,According to the latest SKDS ratings published...,False
2,latvia_tv,https://www.youtube.com/watch?v=-0WQ3-Ql_tc,None,"Mēreni, tā vēlētāju aktivitāti vērtē pieredzēj...",Latvian,False,"Moderately, experienced representatives of the...",False
3,latvia_tv,https://www.youtube.com/watch?v=-0X0eGjKTNc,None,Dodot piekrišanu dzelstaļa izbūvēja saima ir p...,Latvian,False,"Granting consent for the railway construction,...",False
4,latvia_tv,https://www.youtube.com/watch?v=-7PmnNEYIZ8,None,Asbestu saturošie būniecības materiāli Latvijā...,Latvian,True,Asbestos-containing construction materials are...,False
...,...,...,...,...,...,...,...,...
3504,synthetic_news_articles,None,Questioning the Link Between Consumer Behavior...,The topic of climate change has dominated head...,English,True,None,True
3505,synthetic_news_articles,None,Questioning Climate Urgency: The Uncertainty o...,The climate change narrative often paints a di...,English,True,None,True
3506,synthetic_news_articles,None,Rethinking Flooding: Is Climate Change Really ...,Flooding has become a frequent headline in the...,English,True,None,True
3507,synthetic_news_articles,None,The Case for Fossil Fuels: How Energy Independ...,In the face of rapidly changing energy policie...,English,True,None,True


### Classify transcripts with high RECALL

In [4]:
class Classification(BaseModel):
    contains_disinformation: bool
    quotes: list[str]


async def classify_transcripts_high_recall(
    title: str | None, content: str, language: str
) -> Classification:
    system_prompt = f"""You are a specialist in environmental misinformation and climate science. Your task is to critically evaluate the given article for any misleading or inaccurate claims specifically about climate change, energy, biodiversity, and pollution. Focus exclusively on these topics, without considering social or economic dimensions. The article is presented in {language}. After your analysis, present your findings in JSON format.

Here is the JSON format to follow, with examples:

For an article containing misleading statements:

```json
{{
  "containsDisinformation": true,
  "quotes": [
    "Climate change isn't urgent; we have time to adapt.",
    "Technology alone can solve climate change without lifestyle changes.",
    "Plastic recycling is adequate, no need to reduce use.",
    "Increased CO2 benefits agriculture, so climate change could be positive.",
    "Human impact isn't significant, Earth balances itself naturally."
  ]
}}
```

For an article that does not contain misleading information:

```json
{{
  "containsDisinformation": false,
  "quotes": []
}}
```"""

    user_prompt = "Article:\n\n"
    if title:
        user_prompt += f"# {title}\n\n"
    user_prompt += content
    
    try:
        response = await client.beta.chat.completions.parse(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            response_format=Classification,
            temperature=0,
            model="gpt-4o-mini",
        )
        return response.choices[0].message.parsed
    except APITimeoutError:
        # Could retry too, but instead we just ignore it for now
        return Classification(
            contains_disinformation=False,
            quotes=[]
        )



classifications: list[Classification] = await tqdm.gather(
    *[
        classify_transcripts_high_recall(title, content, language)
        for title, content, language in df[["title", "content", "language"]].itertuples(
            index=False
        )
    ],
    desc="Classifying the transcripts",
)
df["y_pred_recall"] = [classif.contains_disinformation for classif in classifications]
df["quotes_recall"] = [classif.quotes for classif in classifications]

Classifying the transcripts: 100%|██████████| 3509/3509 [10:12<00:00,  5.73it/s] 


In [6]:
print(classification_report(df["y_true"], df["y_pred_recall"]))

              precision    recall  f1-score   support

       False       1.00      0.99      0.99      3443
        True       0.58      1.00      0.74        66

    accuracy                           0.99      3509
   macro avg       0.79      0.99      0.87      3509
weighted avg       0.99      0.99      0.99      3509



### Classify transcripts with high PRECISION

In [9]:
df = df[df["y_pred_recall"]].copy()
df

,source,url,title,content,language,is_related_to_climate,translation,y_true,y_pred_recall,quotes_recall
59,latvia_tv,https://www.youtube.com/watch?v=1BHcsgw9tWA,None,Mašā nazeru apkārtnē Ropaža novada Garkalnes p...,Latvian,True,"""In the vicinity of Mašā nazeru, in the Garkal...",False,True,"[Būvniecība plūdu riska dēļ ir pieļaujama, ja ..."
111,latvia_tv,https://www.youtube.com/watch?v=39f82MOv4vI,None,Vairākās Rāznes ezara salās vērojami nobulējuš...,Latvian,True,"On several islands of Lake Rāzna, there are st...",False,True,[Jūras kraukļu dzīvošana rada arī lielu piesār...
353,latvia_tv,https://www.youtube.com/watch?v=awoIElN7PBw,None,Acevišķas ielas Jelgavā pārsauktas par upēm. ...,Latvian,True,Some streets in Jelgava have been renamed afte...,False,True,"[Peldoties pilsē cielās, viss tie paši riski v..."
359,latvia_tv,https://www.youtube.com/watch?v=b2e579feZk4,None,Šogad zēsti 75 meža ugunsgrēki un izdegusi 33 ...,Latvian,True,"This year, 75 forest fires have occurred, and ...",False,True,"[Augstāka temperatūra rada lielāku sausumu, me..."
445,latvia_tv,https://www.youtube.com/watch?v=CNfQHI3Uto8,None,"Lai gan tika solīdz, ka būs ap 20 tūkstošiem p...",Latvian,True,Although it was promised that there would be a...,False,True,[The EU's green policies are extreme and unbea...
...,...,...,...,...,...,...,...,...,...,...
3504,synthetic_news_articles,None,Questioning the Link Between Consumer Behavior...,The topic of climate change has dominated head...,English,True,None,True,True,[Climate cycles are influenced more by natural...
3505,synthetic_news_articles,None,Questioning Climate Urgency: The Uncertainty o...,The climate change narrative often paints a di...,English,True,None,True,True,[The considerable margin of uncertainty means ...
3506,synthetic_news_articles,None,Rethinking Flooding: Is Climate Change Really ...,Flooding has become a frequent headline in the...,English,True,None,True,True,[Attributing every deluge to climate change is...
3507,synthetic_news_articles,None,The Case for Fossil Fuels: How Energy Independ...,In the face of rapidly changing energy policie...,English,True,None,True,True,[The narrative that transitioning to renewable...


In [10]:
class Claim(BaseModel):
    quote: str = Field(
        description="The exact quote from the article corresponding to the claim"
    )
    claim: str = Field(description="The claim that needs human verification")
    context: str = Field(
        description="A reformulation of the context in which this claim was made (maximum 1 paragraph)"
    )
    analysis: str = Field(
        description="Analysis from your expert's perspective on the misinformation of this claim based on the context."
    )


class ClassificationPrecision(BaseModel):
    article_needs_fact_checking: bool
    claims: list[Claim]


async def classify_transcripts_high_precision(
    title: str | None,
    content: str,
    language: str,
    claims: list[str],
) -> Classification:
    system_prompt = f"""You are a context analysis expert specializing in climate change communication. Your task is to evaluate the claims identified in the provided article to determine if they constitute climate change misinformation, considering the full article context. Your objective is to verify the necessity of further fact-checking each claim. Use critical thinking and expert judgment to provide a nuanced analysis. The article is in {language}.

**Input:**
You will receive the full article text followed by a list of claims flagged for potential misinformation. Use the article to assess the context in which each claim was made and determine their validity.

**Input Format:**

```
Article:

<Full article text here>

Claims:

- <List of identified claims>
- ...
```

**Examples of Claims that require additional fact checking:**

- "Climate change isn't urgent; we have time to adapt."
- "Human impact isn't significant; Earth balances itself naturally."

**Examples of Claims that do NOT require additional fact checking (context shows critique or clarification):**

- "Climate change is not caused by man and fighting against it is a way to limit freedom - these are the most common misconceptions about global warming."
- "Some people argue that increased CO2 benefits agriculture, but studies show that the overall impact is detrimental."


**Expected Output Format:**

If the article doesn't require additional fact checking by a human (preferred):

```json
{{
  "article_needs_fact_checking": false,
  "claims": []
}}
```

Otherwise, if the article obviously disinforms about climate change:

```json
{{
  "article_needs_fact_checking": true,
  "claims": [
    {{
      "quote": "Climate change isn't urgent; we have time to adapt.",
      "claim": "The article suggests that addressing climate change is not urgent.",
      "context": "The claim appears as part of an argument questioning the immediate need for climate action within the article narrative.",
      "analysis": "This claim may downplay the urgency required to tackle climate change. Review of the surrounding text suggests it is used to contrast differing views, but the tone suggests potential underestimation of scientific urgency. Further fact-checking is recommended."
    }}
  ]
}}
```

Remember to only flag articles if they deny climate change for sure. Human reviewers will have to spend time reviewing the articles you flag.
"""

    user_prompt = "Article:\n\n"
    if title:
        user_prompt += f"# {title}\n\n"
    user_prompt += content
    user_prompt += "\n\nClaims:\n"
    for claim in claims:
        user_prompt += f"\n- {claim}"

    try:
        response = await client.beta.chat.completions.parse(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            response_format=ClassificationPrecision,
            temperature=0,
            model="gpt-4o-mini",
        )
        return response.choices[0].message.parsed
    except APITimeoutError:
        return ClassificationPrecision(
            article_needs_fact_checking=False,
            claims=[]
        )


classifications: list[ClassificationPrecision] = await tqdm.gather(
    *[
        classify_transcripts_high_precision(title, content, language, quotes)
        for title, content, language, quotes in df[
            ["title", "content", "language", "quotes_recall"]
        ].itertuples(index=False)
    ],
    desc="Classifying the transcripts",
)
df["y_pred_precision"] = [
    classif.article_needs_fact_checking for classif in classifications
]
df["claims_precision"] = [
    [c.model_dump() for c in classif.claims] for classif in classifications
]

Classifying the transcripts: 100%|██████████| 113/113 [00:33<00:00,  3.42it/s]


In [11]:
df

,source,url,title,content,language,is_related_to_climate,translation,y_true,y_pred_recall,quotes_recall,y_pred_precision,claims_precision
59,latvia_tv,https://www.youtube.com/watch?v=1BHcsgw9tWA,None,Mašā nazeru apkārtnē Ropaža novada Garkalnes p...,Latvian,True,"""In the vicinity of Mašā nazeru, in the Garkal...",False,True,"[Būvniecība plūdu riska dēļ ir pieļaujama, ja ...",True,[{'quote': 'Būvniecība plūdu riska dēļ ir pieļ...
111,latvia_tv,https://www.youtube.com/watch?v=39f82MOv4vI,None,Vairākās Rāznes ezara salās vērojami nobulējuš...,Latvian,True,"On several islands of Lake Rāzna, there are st...",False,True,[Jūras kraukļu dzīvošana rada arī lielu piesār...,True,[{'quote': 'Jūras kraukļu dzīvošana rada arī l...
353,latvia_tv,https://www.youtube.com/watch?v=awoIElN7PBw,None,Acevišķas ielas Jelgavā pārsauktas par upēm. ...,Latvian,True,Some streets in Jelgava have been renamed afte...,False,True,"[Peldoties pilsē cielās, viss tie paši riski v...",False,[]
359,latvia_tv,https://www.youtube.com/watch?v=b2e579feZk4,None,Šogad zēsti 75 meža ugunsgrēki un izdegusi 33 ...,Latvian,True,"This year, 75 forest fires have occurred, and ...",False,True,"[Augstāka temperatūra rada lielāku sausumu, me...",True,[{'quote': 'Augstāka temperatūra rada lielāku ...
445,latvia_tv,https://www.youtube.com/watch?v=CNfQHI3Uto8,None,"Lai gan tika solīdz, ka būs ap 20 tūkstošiem p...",Latvian,True,Although it was promised that there would be a...,False,True,[The EU's green policies are extreme and unbea...,True,[{'quote': 'The EU's green policies are extrem...
...,...,...,...,...,...,...,...,...,...,...,...,...
3504,synthetic_news_articles,None,Questioning the Link Between Consumer Behavior...,The topic of climate change has dominated head...,English,True,None,True,True,[Climate cycles are influenced more by natural...,True,[{'quote': 'Climate cycles are influenced more...
3505,synthetic_news_articles,None,Questioning Climate Urgency: The Uncertainty o...,The climate change narrative often paints a di...,English,True,None,True,True,[The considerable margin of uncertainty means ...,True,[{'quote': 'The considerable margin of uncerta...
3506,synthetic_news_articles,None,Rethinking Flooding: Is Climate Change Really ...,Flooding has become a frequent headline in the...,English,True,None,True,True,[Attributing every deluge to climate change is...,True,[{'quote': 'Attributing every deluge to climat...
3507,synthetic_news_articles,None,The Case for Fossil Fuels: How Energy Independ...,In the face of rapidly changing energy policie...,English,True,None,True,True,[The narrative that transitioning to renewable...,True,[{'quote': 'The narrative that transitioning t...


In [12]:
print((df["y_true"] != df["y_pred_precision"]).sum() / len(df))
print(classification_report(df["y_true"], df["y_pred_precision"], zero_division=0))

0.25663716814159293
              precision    recall  f1-score   support

       False       0.95      0.40      0.57        47
        True       0.70      0.98      0.82        66

    accuracy                           0.74       113
   macro avg       0.82      0.69      0.69       113
weighted avg       0.80      0.74      0.71       113



In [13]:
df[df["source"] != "synthetic_news_articles"].to_json(
    "data/df.json", orient="records", indent=2, index=False
)

In [14]:
df_to_review = df[df["y_pred_precision"] & (df["source"] != "synthetic_news_articles")]
df_to_review.to_json(
    "data/articles_to_review.json", orient="records", indent=2, index=False
)